# Video Emotion Recognition Test
This notebook loads your trained model from Google Drive (or local path) and runs inference on a test video.

In [ ]:
# Install requirements if running in Colab
!pip install tensorflow opencv-python-headless numpy

In [ ]:
# Mount Google Drive if you are running this in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except ImportError:
    print("Not running in Colab, skipping Drive mount.")

Mounted at /content/drive
Google Drive mounted.


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from IPython.display import HTML
from base64 import b64encode
import os

# Path to your saved model in Google Drive (or local path)
# Update this if your model is named differently or saved in a different folder
MODEL_PATH = "/content/drive/MyDrive/PersonaPath/checkpoints/cnn_emotion_phase4.keras"

print(f"Loading model from {MODEL_PATH}...")
if not os.path.exists(MODEL_PATH):
    print(f"WARNING: Model not found at {MODEL_PATH}. Please check the path.")
else:
    model = tf.keras.models.load_model(MODEL_PATH, safe_mode=False)
    print("Model loaded successfully!")

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Load Haar cascade for face detection
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

Loading model from /content/drive/MyDrive/PersonaPath/checkpoints/cnn_emotion_phase4.keras...
Model loaded successfully!


In [ ]:
# Function to process video
def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("Error opening video file")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5, minSize=(60, 60))

        for (x, y, w, h) in faces:
            face_img = frame[y:y+h, x:x+w]
            try:
                face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
                face_resized = cv2.resize(face_rgb, (224, 224))
                input_tensor = np.expand_dims(face_resized, axis=0)

                preds = model.predict(input_tensor, verbose=0)
                emotion_idx = np.argmax(preds[0])
                emotion = emotion_labels[emotion_idx]
                confidence = preds[0][emotion_idx]

                color = (0, 255, 0)
                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
                text = f"{emotion} ({confidence:.2f})"
                cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
            except Exception as e:
                pass

        out.write(frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"Processed {frame_count} frames...")

    cap.release()
    out.release()
    print("Processing complete!")

# Run it on a test video
INPUT_VIDEO = "/content/Test_emotion_vid1.mp4" # Upload a video and change this path
OUTPUT_VIDEO = "output_emotion.mp4"

if os.path.exists(INPUT_VIDEO):
    process_video(INPUT_VIDEO, OUTPUT_VIDEO)
else:
    print(f"Please upload a video named {INPUT_VIDEO} to the current directory.")

Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processing complete!


In [ ]:
# Display the output video
# If the video doesn't play in Colab, you can download 'output_emotion.mp4' directly.
if os.path.exists(OUTPUT_VIDEO):
    # Try converting to h264 for web playback
    !ffmpeg -y -i {OUTPUT_VIDEO} -vcodec libx264 temp_h264.mp4 -hide_banner -loglevel error
    if os.path.exists("temp_h264.mp4"):
        os.replace("temp_h264.mp4", OUTPUT_VIDEO)

    mp4 = open(OUTPUT_VIDEO, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width="640" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))